# TableGuard-Lite — start here
**Phase 1: hardware, public-data inspection, and supervisor tests.**

No GPU or pretrained model is required for the offline cells. Keep this notebook inside the extracted `TableGuard_Starter/notebooks/` folder and select your TableGuard Python environment.

This notebook does **not** implement the required dual-SO-101 policy, object perception, or robot recovery. Hand-authored logic fixtures are labeled as such. Public downloads are opt-in and never substituted with generated data.

Source details: `docs/SOURCES.md`. Exact validation scope: `docs/VALIDATION.md`.

## 0. Keep every output inside the project folder

In [ ]:
from pathlib import Path
import sys, os, json, subprocess
ROOT = Path.cwd().resolve()
while not (ROOT / "tableguard" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "tableguard" / "__init__.py").exists():
    raise RuntimeError("Place this notebook in the extracted TableGuard_Starter/notebooks folder first.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("No files will be uploaded by the hardware or offline test steps.")

## 1. Inspect your actual hardware
The report does not certify track eligibility. NVIDIA VRAM and system RAM are different quantities. An absent `nvidia-smi` command does not prove there is no GPU. Framework discovery is optional; it does not benchmark a model.

In [ ]:
from tableguard.hardware import inspect_hardware
from tableguard.common import write_json
report = inspect_hardware(probe_frameworks=False)
write_json(ROOT / "artifacts/hardware_report.json", report)
for key in ["cpu_model", "logical_cpu_count", "ram_total_gib", "ram_available_gib", "nvidia", "graphics_devices", "eligibility"]:
    if key in report:
        print(key + ":", report[key])
print("Saved artifacts/hardware_report.json")

## 2. Run the offline unit tests
These test supplied goal states, retry limits, stale/unknown observations and safe dataset paths. Downloader tests use mocks. Passing is **not** evidence of successful robot episodes.

In [ ]:
p = subprocess.run([sys.executable, "-m", "tableguard", "selftest"], cwd=ROOT,
                   capture_output=True, text=True)
print(p.stdout)
print(p.stderr)
if p.returncode != 0:
    raise RuntimeError("A unit test failed. Fix it before connecting the robot adapter.")
summary = json.loads((ROOT / "artifacts/unit_test_summary.json").read_text())
summary

## 3. Inspect decision logic on hand-authored fixtures
These are **structured test inputs**, not camera predictions. `repair` means a request; no robot is connected and nothing is moved.

In [ ]:
from tableguard.fixtures import run_fixture_demo
from IPython.display import display, HTML
import html
fixture_report = run_fixture_demo()
write_json(ROOT / "artifacts/logic_fixture_results.json", fixture_report)
rows = []
for row in fixture_report["decisions"]:
    vals = [row["fixture"], row["response"], ", ".join(row["affected_ids"]), row["reason"]]
    rows.append("<tr>" + "".join("<td>" + html.escape(str(v)) + "</td>" for v in vals) + "</tr>")
display(HTML("<b>LOGIC FIXTURES — NOT ROBOT RESULTS</b><table><tr><th>Fixture</th><th>Request</th><th>Affected goals</th><th>Reason</th></tr>" + "".join(rows) + "</table>"))
print("Robot executed:", fixture_report["robot_executed"])

## 4. Choose ONE public reference
Start with `lerobot/svla_so101_pickplace` for a small I/O test. Publisher metadata says `so100_follower`, despite its name. `gpudad/so101_pick_cube` is an optional single-arm MuJoCo reference. The ALOHA sample has a different action space. None is the final required bimanual table-setting dataset.

In [ ]:
catalog = json.loads((ROOT / "configs/datasets.json").read_text())
for item in catalog["datasets"]:
    print(item["repo_id"])
    print("  ", item["use"])
    print("  ", item["warning"])
    print("  ", item["url"])

## 5. Download metadata and one episode only when ready
Requires internet and `requirements-phase1.txt`. Set the flag below to `True`. The downloader resolves a fixed revision, checks sizes, limits the total to 50 MB, and saves provenance. It downloads neither a model nor the whole dataset.

V3 media extraction is intentionally unsupported rather than guessing episode offsets. For the v3 simulation reference, use metadata-only inspection first.

In [ ]:
DOWNLOAD_PUBLIC_SAMPLE = False  # Change to True on your connected development computer.
PUBLIC_REPO = "lerobot/svla_so101_pickplace"
manifest_path = None
if DOWNLOAD_PUBLIC_SAMPLE:
    from tableguard.datasets import fetch_reference
    manifest_path = fetch_reference(PUBLIC_REPO, episode=0, include_media=True, max_mb=50)
    write_json(ROOT / "artifacts/latest_public_sample.json", {"manifest": str(manifest_path.relative_to(ROOT))})
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps(manifest["metadata_summary"], indent=2))
    print("Download manifest:", manifest_path)
else:
    print("Public download not requested. No dataset or model has been downloaded by this cell.")

## 6. Visualize the downloaded reference and save every figure
The plots show recorded public frames and actions—not predictions. No output is fabricated when a download or decoder fails. Camera/row timing alignment and action units still require inspection of the actual source metadata.

In [ ]:
if manifest_path is not None:
    from tableguard.visualize import visualize_sample
    from IPython.display import Image
    visual_report = visualize_sample(manifest_path)
    for image_path in visual_report["images"]:
        display(Image(filename=str(ROOT / image_path)))
    if visual_report["action_plot"]:
        display(Image(filename=str(ROOT / visual_report["action_plot"])))
    print(json.dumps(visual_report, indent=2))
else:
    print("No live sample in this notebook run. Enable the preceding download cell to inspect real public data.")

## 7. Save the environment and report next-stage blockers
The package listing records this kernel, not a verified target-machine environment. Do not submit an assistant/preparation-runtime report as your Intel hardware evidence.

In [ ]:
p = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
if p.returncode == 0:
    (ROOT / "artifacts/environment_freeze.txt").write_text(p.stdout, encoding="utf-8")
    print("Saved artifacts/environment_freeze.txt")
else:
    print("Could not record pip environment:", p.stderr)
print("Next: obtain approved scene/policy, actual camera and joint-action contracts, and target Intel execution.")
print("Share your hardware_report.json and unit_test_summary.json for the next implementation step.")

## Next integration gate
M2 must prove the approved dual-arm baseline; M3 must implement a real camera-based goal checker; M1/M2 connect a supported corrective action. The current `UnconfiguredRobotAdapter` deliberately raises an error rather than pretending to execute a repair.

The immediate milestone is a runnable baseline on eligible hardware. The planned 12 held-out cases / 36 robot episodes are **not** the 49 software unit tests above. No training or benchmark accuracy is claimed here.